---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Homework 5**: RAG System for Fordham University

### 📅 **Due Date**: Day of Lecture 7, 11:59 PM

### Difficulty: ★★★★☆


**Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

In this homework, you'll complete the RAG (Retrieval Augmented Generation) system you started building in Lecture 5. You will build an end-to-end pipeline that can answer questions about Fordham University using real data scraped from the Fordham website.

This is an open-ended assignment — there is no single right implementation, and you're encouraged to experiment with chunking strategies, embedding models, prompt design, and retrieval parameters to improve your system. **I will grade your system by testing it with specific questions that I know the answer to.**

---

## Instructions

- You may use ChatGPT, Claude, documentation, Stack Overflow, etc. When using external resources, briefly cite them in a comment.
- Your submission must include **pre-computed embeddings** and any other artifacts needed so that I can run your RAG system without recomputing anything expensive. Share it with us in a way that makes sense.
- Run all cells before submitting to ensure they work.

**Submission:**
1. Create a branch called `homework-5`
2. Commit and push your work (notebook + Streamlit app + saved embeddings/artifacts in `temp/`)
3. Create a PR and merge to main
4. Submit the `.ipynb` file on Blackboard

---

## Step 1: Load and Chunk the Fordham Website Data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more. The first line of every file is the **URL** of the page it was scraped from. The rest is the page content in Markdown.

Think about: chunk size, what to split on (paragraphs, headers, fixed length, etc.), whether chunks should overlap, and how to track which page each chunk came from.

In [1]:
pip install tqdm


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# YOUR CODE HERE\
import os
import zipfile
import re
from pathlib import Path
from tqdm import tqdm

# ── Config ──────────────────────────────────────────────────────────────────
ZIP_PATH      = r"ce/data/fordham-website-windows.zip"
CHUNK_SIZE    = 500   # target words per chunk
CHUNK_OVERLAP = 50    # words of overlap between chunks

# ── 1. Unzip (only once) ────────────────────────────────────────────────────
extract_dir = Path(ZIP_PATH).parent / "fordham-website"
if not extract_dir.exists():
    print("Unzipping... (this may take a minute)")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(extract_dir)
    print(f"Extracted to {extract_dir}")
else:
    print(f"Already extracted: {extract_dir}")

# ── 2. Discover all markdown files ──────────────────────────────────────────
md_files = list(extract_dir.rglob("*.md"))
print(f"Found {len(md_files):,} markdown files")

# ── 3. Chunking helper ───────────────────────────────────────────────────────
def chunk_text(text: str, chunk_size: int, overlap: int) -> list[str]:
    """Split text into overlapping word-based chunks."""
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

# ── 4. Load files and build chunks ──────────────────────────────────────────
all_chunks = []  # list of dicts: {text, source_url, file_path}

for fpath in tqdm(md_files, desc="Loading files"):
    try:
        content = fpath.read_text(encoding="utf-8", errors="ignore").strip()
    except Exception:
        continue

    if not content:
        continue

    # First line is the URL
    lines = content.split("\n")
    source_url = lines[0].strip() if lines else str(fpath)
    body = "\n".join(lines[1:]).strip()

    if len(body.split()) < 20:   # skip near-empty pages
        continue

    for chunk in chunk_text(body, CHUNK_SIZE, CHUNK_OVERLAP):
        if len(chunk.split()) < 20:   # skip tiny trailing chunks
            continue
        all_chunks.append({
            "text": chunk,
            "source_url": source_url,
            "file_path": str(fpath.name),
        })

print(f"\nTotal chunks created: {len(all_chunks):,}")
print(f"\nExample chunk:\n{'-'*60}")
print(f"URL : {all_chunks[0]['source_url']}")
print(f"Text: {all_chunks[0]['text'][:300]}...")

Unzipping... (this may take a minute)
Extracted to ce\data\fordham-website
Found 9,560 markdown files


Loading files: 100%|██████████| 9560/9560 [03:23<00:00, 46.95it/s] 


Total chunks created: 16,878

Example chunk:
------------------------------------------------------------
URL : https://www.fordham.edu/about/living-the-mission/center-on-religion-and-culture/duffy-fellows-program/past-duffy-fellows/2021-2022-duffy-fellows
Text: # 2021-2022 Duffy Fellows **Afrah Bandagi (FCLC 2023)** “[Supera las fronteras (Transcend Borders): Spirituality and Migration Activism](https://youtu.be/FHzAAH6iP40?si=wvNcQl9Tlman6nRZ)” (research presentation) **Major:** Philosophy and Political Science **Minor:** Peace and Justice Studies Afrah B...


---

## Step 2: Embed the Chunks

Turn each chunk into a vector so you can search over them. You can use a local model or an API model — your choice.

Once you've created your embeddings, **save them somewhere** so you (and I) don't have to redo this step. Save the chunk metadata too (text, source URL, etc.).

In [3]:
pip install openai


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install tiktoken

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
# Clean up old progress files
import os
for f in ["temp/embeddings_partial.npy", "temp/progress_idx.json"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"Deleted {f}")
print("Ready to restart embedding from scratch")

Deleted temp/embeddings_partial.npy
Deleted temp/progress_idx.json
Ready to restart embedding from scratch


In [9]:
# YOUR CODE HERE
import json
import numpy as np
import time
import tiktoken
from openai import OpenAI
from pathlib import Path
from tqdm import tqdm
from dotenv import load_dotenv

load_dotenv()

# ── Config ───────────────────────────────────────────────────────────────────
EMBED_MODEL     = "text-embedding-3-small"
MAX_TOKENS      = 8000   # safely under the 8192 limit
BATCH_SIZE      = 100
SAVE_DIR        = Path("temp")
SAVE_DIR.mkdir(exist_ok=True)

EMBEDDINGS_PATH = SAVE_DIR / "embeddings.npy"
CHUNKS_PATH     = SAVE_DIR / "chunks.json"
PROGRESS_PATH   = SAVE_DIR / "embeddings_partial.npy"
PROGRESS_IDX    = SAVE_DIR / "progress_idx.json"

client   = OpenAI()
tokenizer = tiktoken.get_encoding("cl100k_base")

# ── Truncate to exactly MAX_TOKENS ───────────────────────────────────────────
def truncate_to_tokens(text: str, max_tokens: int = MAX_TOKENS) -> str:
    tokens = tokenizer.encode(text)
    if len(tokens) <= max_tokens:
        return text
    return tokenizer.decode(tokens[:max_tokens])

# ── Embed one batch safely ────────────────────────────────────────────────────
def embed_batch(texts: list[str]) -> list[list[float]]:
    texts = [truncate_to_tokens(t) for t in texts]
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=texts
    )
    return [item.embedding for item in response.data]

# ── Check if fully done ───────────────────────────────────────────────────────
if EMBEDDINGS_PATH.exists() and CHUNKS_PATH.exists():
    print("✅ Embeddings already fully saved — loading from disk...")
    embeddings = np.load(EMBEDDINGS_PATH)
    with open(CHUNKS_PATH, "r", encoding="utf-8") as f:
        all_chunks = json.load(f)
    print(f"Loaded {len(all_chunks):,} chunks, embeddings shape={embeddings.shape}")

else:
    # ── Resume from partial progress if available ─────────────────────────────
    if PROGRESS_PATH.exists() and PROGRESS_IDX.exists():
        all_embeddings = np.load(PROGRESS_PATH).tolist()
        with open(PROGRESS_IDX, "r") as f:
            start_i = json.load(f)["next_i"]
        print(f"▶️  Resuming from chunk {start_i} ({len(all_embeddings):,} embeddings already done)")
    else:
        all_embeddings = []
        start_i = 0
        print(f"Starting fresh — embedding {len(all_chunks):,} chunks...")

    for i in tqdm(range(0, len(all_chunks), BATCH_SIZE), desc="Embedding batches"):
        # Skip already completed batches
        if i + BATCH_SIZE <= start_i:
            continue

        batch = all_chunks[i : i + BATCH_SIZE]
        texts = [c["text"] for c in batch]

        try:
            vecs = embed_batch(texts)
            all_embeddings.extend(vecs)
        except Exception as e:
            print(f"\nError at chunk {i}: {e}")
            print("Saving progress and stopping — re-run to resume.")
            np.save(PROGRESS_PATH, np.array(all_embeddings, dtype=np.float32))
            with open(PROGRESS_IDX, "w") as f:
                json.dump({"next_i": i}, f)
            raise

        # Save progress every 20 batches
        if (i // BATCH_SIZE) % 20 == 0:
            np.save(PROGRESS_PATH, np.array(all_embeddings, dtype=np.float32))
            with open(PROGRESS_IDX, "w") as f:
                json.dump({"next_i": i + BATCH_SIZE}, f)

        time.sleep(0.05)

    # ── All done — save final ─────────────────────────────────────────────────
    embeddings = np.array(all_embeddings, dtype=np.float32)
    np.save(EMBEDDINGS_PATH, embeddings)
    with open(CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(all_chunks, f, ensure_ascii=False)

    # Clean up partial files
    if PROGRESS_PATH.exists(): PROGRESS_PATH.unlink()
    if PROGRESS_IDX.exists():  PROGRESS_IDX.unlink()

    print(f"\n✅ Saved embeddings: {EMBEDDINGS_PATH}  shape={embeddings.shape}")
    print(f"✅ Saved chunks:     {CHUNKS_PATH}")

Starting fresh — embedding 16,878 chunks...


Embedding batches: 100%|██████████| 169/169 [10:08<00:00,  3.60s/it]



✅ Saved embeddings: temp\embeddings.npy  shape=(16878, 1536)
✅ Saved chunks:     temp\chunks.json


---

## Step 3: Retrieve

Build the **R** in RAG. Write a function that takes a question and returns the most relevant chunks. You can use semantic search, BM25, hybrid — whatever you think works best.

Test it on a few questions and eyeball whether the results make sense.

In [10]:
# YOUR CODE HERE
import numpy as np
import json

# ── Cosine similarity search ──────────────────────────────────────────────────
def get_query_embedding(question: str) -> np.ndarray:
    response = client.embeddings.create(
        model=EMBED_MODEL,
        input=[truncate_to_tokens(question)]
    )
    return np.array(response.data[0].embedding, dtype=np.float32)

def retrieve(question: str, top_k: int = 5) -> list[dict]:
    query_vec = get_query_embedding(question)

    # Cosine similarity = dot product on normalized vectors
    norms = np.linalg.norm(embeddings, axis=1)
    query_norm = np.linalg.norm(query_vec)
    similarities = (embeddings @ query_vec) / (norms * query_norm + 1e-10)

    # Get top_k indices
    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = []
    for idx in top_indices:
        results.append({
            "text": all_chunks[idx]["text"],
            "source_url": all_chunks[idx]["source_url"],
            "score": float(similarities[idx])
        })
    return results

# ── Test it ───────────────────────────────────────────────────────────────────
test_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid at Fordham?",
    "What is the tuition for undergraduate students?",
]

for q in test_questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    results = retrieve(q, top_k=3)
    for i, r in enumerate(results, 1):
        print(f"\n[{i}] Score: {r['score']:.4f}")
        print(f"    URL: {r['source_url']}")
        print(f"    Text: {r['text'][:200]}...")


Q: What programs does the Gabelli School of Business offer?

[1] Score: 0.7595
    URL: https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions
    Text: # Gabelli Academic Programs and Admissions Our wide-ranging business school programs won’t just shape your career – they’ll offer a transformative experience that will leave you ready to use business ...

[2] Score: 0.7594
    URL: https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs/academic-programs
    Text: # Gabelli Graduate Academic Programs ![Academic Programs](/media/home/schools/gabelli-school-of-business/gsb_academic_programs.jpg) No matter what your career goal, Fordham has a business program tail...

[3] Score: 0.7559
    URL: https://www.fordham.edu/gabelli-school-of-business
    Text: # Gabelli School of Business ## Graduate Business Programs – Advance Your Career The Gabelli School’s nationally and globally ranked graduate business degrees foc

---

## Step 4: Generate

Build the **G** in RAG. Write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.

Think about: how to structure the prompt, what the LLM should do when the context doesn't contain the answer, and which model to use.

In [16]:
# ── Generate answer using retrieved context ───────────────────────────────────

def generate(question: str, retrieved_chunks: list[dict]) -> str:
    # Build context from retrieved chunks
    context = ""
    for i, chunk in enumerate(retrieved_chunks, 1):
        context += f"[Source {i} - {chunk['source_url']}]:\n{chunk['text']}\n\n"

    prompt = f"""You are a helpful assistant for Fordham University. 
Answer the user's question using ONLY the provided context below.
If the context does not contain enough information to answer the question, 
say "I don't have enough information to answer that question" — do not make anything up.
At the end of your answer, list the full URLs of the sources you used under a "Sources:" section.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a helpful Fordham University assistant. Answer questions accurately based only on the provided context."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2,
        max_tokens=600
    )

    return response.choices[0].message.content


# ── Example ───────────────────────────────────────────────────────────────────
example_question = "How do I apply for financial aid at Fordham?"
example_chunks   = retrieve(example_question, top_k=3)
example_answer   = generate(example_question, example_chunks)

print(f"Q: {example_question}")
print(f"\nA: {example_answer}")


Q: How do I apply for financial aid at Fordham?

A: To apply for financial aid at Fordham University, you should fill out both the Free Application for Federal Student Aid (FAFSA) to be considered for federal aid and the CSS Profile to be considered for institutional aid. Deadlines for these applications vary based on your choice of admission application (Early Action, Early Decision, or Regular Decision). Once you are admitted, Fordham will use the information from these applications to create a financial aid package for you.

For more detailed information, you can visit the financial aid section of Fordham's website or contact the Office of Undergraduate Admission.

Sources:
- https://www.fordham.edu/undergraduate-admission/admitted-students/financial-aid
- https://www.fordham.edu/families/i-am-a-parent-of/prospective-students


---

## Step 5: Wire it Together

Combine the previous steps into a single `rag(question)` function. Question in, answer out.

In [18]:
# YOUR CODE HERE
def rag(question: str, top_k: int = 5) -> str:
    retrieved_chunks = retrieve(question, top_k=top_k)
    answer = generate(question, retrieved_chunks)
    return answer

In [19]:
# Demonstrate your RAG system

demo_questions = [
    "What programs does the Gabelli School of Business offer?",
    "How do I apply for financial aid at Fordham?",
    "What is the tuition for undergraduate students?",
    "Tell me about Fordham's campus locations.",
    "What research opportunities are available for students?",
]

for q in demo_questions:
    print(f"Q: {q}")
    answer = rag(q)
    print(f"A: {answer}")
    print("-" * 80)

Q: What programs does the Gabelli School of Business offer?
A: The Gabelli School of Business offers a variety of programs including three variants of the MBA program: a full-time cohort MBA, a Professional MBA (which includes part-time options), and an Executive MBA. Additionally, there are 13 specialized Master of Science programs tailored to various business niches, such as Accounting, Artificial Intelligence in Business, Business Analytics, Finance, Information Technology, Management, Marketing Intelligence, Media Management, Quantitative Finance, and Taxation.

Sources:
- https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions
- https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs/academic-programs
- https://www.fordham.edu/gabelli-school-of-business/academic-programs-and-admissions/graduate-programs
--------------------------------------------------------------------------------
Q: How do I apply for fin

---

## Step 6: Evaluate Your RAG System

A working RAG system is great — but how do you know it's actually *good*? You can't improve what you can't measure. In this step you'll build an evaluation framework using concepts from Lecture 6.

There are two things to evaluate in a RAG system:
- **Retrieval quality**: Are you finding the right chunks?
- **Answer quality**: Is the generated answer correct and grounded in the context?

### Build a test set

Create a test set of at least **10 question-answer pairs**. For each pair, provide the question and the expected answer (look it up in the data). Cover a range of question types — factual, procedural, about specific programs, etc.

### Evaluate retrieval

For each question, check whether the retrieved chunks actually contain relevant information. You can do this manually or automatically (e.g., use an LLM to judge relevance). Compute **context relevance** — the fraction of retrieved chunks that are actually useful.

### Evaluate answers with LLM-as-judge

Use an LLM to evaluate your system's answers on two dimensions:

1. **Faithfulness**: Does the answer only use information from the retrieved context? (No hallucination)
2. **Correctness**: Is the answer factually correct compared to the expected answer?

Use **structured outputs** (Pydantic) to get consistent scores from the judge. A starting schema is provided below — feel free to modify it.

In [23]:
from pydantic import BaseModel, Field

class RAGEvaluation(BaseModel):
    faithfulness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely hallucinated, 5=fully grounded in context"
    )
    faithfulness_reasoning: str = Field(
        ..., description="Brief explanation of the faithfulness score"
    )
    correctness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely wrong, 5=fully correct and complete"
    )
    correctness_reasoning: str = Field(
        ..., description="Brief explanation of the correctness score"
    )

In [24]:
# YOUR CODE HERE
# - Build your test set
# - Evaluate retrieval quality (context relevance)
# - Evaluate answer quality with LLM-as-judge (faithfulness + correctness)
# - Summarize your results

from pydantic import BaseModel, Field
from openai import OpenAI

# ── Evaluation schema ─────────────────────────────────────────────────────────
class RAGEvaluation(BaseModel):
    faithfulness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely hallucinated, 5=fully grounded in context"
    )
    faithfulness_reasoning: str = Field(
        ..., description="Brief explanation of the faithfulness score"
    )
    correctness_score: int = Field(
        ..., ge=1, le=5,
        description="1=completely wrong, 5=fully correct and complete"
    )
    correctness_reasoning: str = Field(
        ..., description="Brief explanation of the correctness score"
    )

# ── Test set: 10 question-answer pairs ───────────────────────────────────────
test_set = [
    {
        "question": "What programs does the Gabelli School of Business offer?",
        "expected": "MBA (full-time, professional, executive), 13 MS programs including Accounting, Finance, Marketing Intelligence, Business Analytics, and more."
    },
    {
        "question": "How do I apply for financial aid at Fordham?",
        "expected": "Fill out the FAFSA for federal aid and the CSS Profile for institutional aid. Deadlines vary by admission type."
    },
    {
        "question": "Where is Fordham's Rose Hill campus located?",
        "expected": "441 E. Fordham Rd., Bronx, NY 10458. It is an 85-acre campus adjacent to the New York Botanical Garden and across from the Bronx Zoo."
    },
    {
        "question": "What is the Lincoln Center campus near?",
        "expected": "Located at 113 West 60th Street in Manhattan, directly across from Lincoln Center for the Performing Arts and close to Central Park."
    },
    {
        "question": "What undergraduate research opportunities exist at Fordham?",
        "expected": "Students can do faculty research, apply for summer research grants, present at symposiums, and work at places like Memorial Sloan Kettering."
    },
    {
        "question": "What is the summer session undergraduate tuition per credit?",
        "expected": "$1,134 per credit for undergraduate students in the 2026 summer session."
    },
    {
        "question": "What is the Westchester campus address?",
        "expected": "400 Westchester Avenue, West Harrison, NY 10604. It has a three-story building on 32 landscaped acres."
    },
    {
        "question": "What is the Duffy Fellows program?",
        "expected": "A fellowship program at Fordham's Center on Religion and Culture where students conduct research on topics related to spirituality and society."
    },
    {
        "question": "Does Fordham have a law school and where is it?",
        "expected": "Yes, the School of Law is located at the Lincoln Center campus in Manhattan."
    },
    {
        "question": "What federal forms are needed for financial aid?",
        "expected": "The FAFSA (Free Application for Federal Student Aid) and the CSS Profile for institutional aid."
    },
]

# ── LLM-as-judge evaluator ────────────────────────────────────────────────────
def evaluate(question: str, expected: str, actual_answer: str, context: str) -> RAGEvaluation:
    prompt = f"""You are an expert evaluator for a RAG system. Evaluate the answer below on two dimensions.

QUESTION: {question}

EXPECTED ANSWER: {expected}

RETRIEVED CONTEXT:
{context}

ACTUAL ANSWER: {actual_answer}

Score on:
1. FAITHFULNESS (1-5): Does the answer only use info from the retrieved context? No hallucination?
2. CORRECTNESS (1-5): Is the answer factually correct compared to the expected answer?
"""
    response = client.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a strict but fair RAG evaluator."},
            {"role": "user", "content": prompt}
        ],
        response_format=RAGEvaluation,
        temperature=0
    )
    return response.choices[0].message.parsed

# ── Evaluate context relevance ────────────────────────────────────────────────
def context_relevance(question: str, chunks: list[dict]) -> float:
    scores = []
    for chunk in chunks:
        prompt = f"""Is the following text relevant to answering this question?
Question: {question}
Text: {chunk['text'][:500]}
Reply with only 1 (relevant) or 0 (not relevant)."""
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
            max_tokens=5
        )
        reply = response.choices[0].message.content.strip()
        scores.append(1 if "1" in reply else 0)
    return sum(scores) / len(scores) if scores else 0.0

# ── Run full evaluation ───────────────────────────────────────────────────────
print("Running evaluation on 10 test questions...\n")

results = []
for item in test_set:
    q        = item["question"]
    expected = item["expected"]

    # Retrieve and generate
    chunks  = retrieve(q, top_k=5)
    answer  = generate(q, chunks)
    context = "\n\n".join([c["text"] for c in chunks])

    # Score
    eval_result  = evaluate(q, expected, answer, context)
    ctx_relevance = context_relevance(q, chunks)

    results.append({
        "question":           q,
        "expected":           expected,
        "actual":             answer,
        "faithfulness":       eval_result.faithfulness_score,
        "faithfulness_reason": eval_result.faithfulness_reasoning,
        "correctness":        eval_result.correctness_score,
        "correctness_reason": eval_result.correctness_reasoning,
        "context_relevance":  ctx_relevance,
    })

    print(f"Q: {q}")
    print(f"  Faithfulness : {eval_result.faithfulness_score}/5 — {eval_result.faithfulness_reasoning}")
    print(f"  Correctness  : {eval_result.correctness_score}/5 — {eval_result.correctness_reasoning}")
    print(f"  Ctx Relevance: {ctx_relevance:.0%}")
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
avg_faith = sum(r["faithfulness"] for r in results) / len(results)
avg_corr  = sum(r["correctness"]  for r in results) / len(results)
avg_ctx   = sum(r["context_relevance"] for r in results) / len(results)

print("=" * 60)
print("EVALUATION SUMMARY")
print("=" * 60)
print(f"Avg Faithfulness     : {avg_faith:.2f} / 5")
print(f"Avg Correctness      : {avg_corr:.2f} / 5")
print(f"Avg Context Relevance: {avg_ctx:.0%}")

Running evaluation on 10 test questions...

Q: What programs does the Gabelli School of Business offer?
  Faithfulness : 5/5 — The actual answer accurately reflects the information provided in the retrieved context without introducing any hallucinations or inaccuracies. It uses specific details from the context regarding the MBA and MS programs offered by the Gabelli School of Business.
  Correctness  : 5/5 — The actual answer is factually correct and complete, matching the expected answer by listing the three variants of the MBA program and providing a comprehensive list of the 13 specialized Master of Science programs.
  Ctx Relevance: 100%

Q: How do I apply for financial aid at Fordham?
  Faithfulness : 5/5 — The actual answer accurately reflects the information provided in the retrieved context without introducing any hallucinations or inaccuracies. It correctly mentions the FAFSA and CSS Profile as the necessary applications for financial aid and notes the variation in deadlines 

---

## Step 7: Build a Streamlit App

Your RAG system lives inside a notebook — that's great for development, but nobody is going to use a Jupyter notebook to ask questions about Fordham. Turn it into a web app using [Streamlit](https://docs.streamlit.io/).

Create a `.py` file (e.g., `scripts/fordham_rag_app.py`) that:
1. Lets the user type a question about Fordham
2. Runs your RAG pipeline
3. Displays the answer and the source pages used

**Getting started:**
- Install: `uv pip install streamlit`
- Run: `streamlit run scripts/fordham_rag_app.py`

**Tip**: Use `@st.cache_resource` to avoid reloading embeddings on every interaction.

**Include a screenshot of your working app below.**

*Paste your screenshot here*

## Step 8: How to Run Your System

| Item | Your Answer |
|------|-------------|
| **Embedding model used** | `text-embedding-3-small` (OpenAI) |
| **LLM used for generation** | `gpt-4o-mini` (OpenAI) |
| **LLM used for evaluation (judge)** | `gpt-4o-mini` (OpenAI) |
| **Saved artifacts** | `temp/embeddings.npy`, `temp/chunks.json` |
| **How to start the Streamlit app** | `streamlit run scripts/fordham_rag_app.py` |
| **Any API keys or env vars needed** | `OPENAI_API_KEY` in `.env` file at project root |
| **Anything else I should know** | Run all notebook cells top to bottom before using the app. Embeddings are pre-computed and saved in `temp/` — no need to re-embed. |


---

## Bonus: Experiment and Improve

Now that you have a working RAG system *and* a way to measure its quality, try to improve it. Use your evaluation framework to measure the impact of changes.

Ideas: different chunk sizes, different embedding models, hybrid search, better prompts, reranking, query rewriting. Document what you tried and show before/after evaluation scores.

In [28]:
# YOUR CODE HERE
# ── BONUS: Experiment with smaller chunk size ─────────────────────────────────
# Hypothesis: smaller chunks (250 words) = more precise retrieval = better scores

from pathlib import Path
import zipfile, json, numpy as np, time, tiktoken
from tqdm import tqdm

BONUS_CHUNK_SIZE    = 250
BONUS_CHUNK_OVERLAP = 25
BONUS_SAVE_DIR      = Path("temp")
BONUS_EMB_PATH      = BONUS_SAVE_DIR / "embeddings_bonus.npy"
BONUS_CHUNKS_PATH   = BONUS_SAVE_DIR / "chunks_bonus.json"

# ── Re-chunk with smaller size ────────────────────────────────────────────────
if not BONUS_CHUNKS_PATH.exists():
    print("Re-chunking with smaller chunk size...")
    extract_dir = Path("ce/data/fordham-website")
    md_files    = list(extract_dir.rglob("*.md"))

    bonus_chunks = []
    for fpath in tqdm(md_files, desc="Chunking"):
        try:
            content = fpath.read_text(encoding="utf-8", errors="ignore").strip()
        except:
            continue
        if not content:
            continue
        lines      = content.split("\n")
        source_url = lines[0].strip()
        body       = "\n".join(lines[1:]).strip()
        if len(body.split()) < 20:
            continue
        for chunk in chunk_text(body, BONUS_CHUNK_SIZE, BONUS_CHUNK_OVERLAP):
            if len(chunk.split()) < 15:
                continue
            bonus_chunks.append({"text": chunk, "source_url": source_url})

    with open(BONUS_CHUNKS_PATH, "w", encoding="utf-8") as f:
        json.dump(bonus_chunks, f, ensure_ascii=False)
    print(f"Created {len(bonus_chunks):,} bonus chunks")
else:
    with open(BONUS_CHUNKS_PATH, "r", encoding="utf-8") as f:
        bonus_chunks = json.load(f)
    print(f"Loaded {len(bonus_chunks):,} bonus chunks")

# ── Re-embed ──────────────────────────────────────────────────────────────────
if not BONUS_EMB_PATH.exists():
    print("Embedding bonus chunks...")
    all_emb = []
    for i in tqdm(range(0, len(bonus_chunks), 100), desc="Embedding"):
        batch = [truncate_to_tokens(c["text"]) for c in bonus_chunks[i:i+100]]
        resp  = client.embeddings.create(model=EMBED_MODEL, input=batch)
        all_emb.extend([r.embedding for r in resp.data])
        time.sleep(0.05)
    bonus_embeddings = np.array(all_emb, dtype=np.float32)
    np.save(BONUS_EMB_PATH, bonus_embeddings)
    print(f"Saved bonus embeddings: {bonus_embeddings.shape}")
else:
    bonus_embeddings = np.load(BONUS_EMB_PATH)
    print(f"Loaded bonus embeddings: {bonus_embeddings.shape}")

# ── Bonus retrieve function ───────────────────────────────────────────────────
def retrieve_bonus(question: str, top_k: int = 5) -> list[dict]:
    query_vec    = get_query_embedding(question)
    norms        = np.linalg.norm(bonus_embeddings, axis=1)
    query_norm   = np.linalg.norm(query_vec)
    similarities = (bonus_embeddings @ query_vec) / (norms * query_norm + 1e-10)
    top_indices  = np.argsort(similarities)[::-1][:top_k]
    return [
        {
            "text":       bonus_chunks[i]["text"],
            "source_url": bonus_chunks[i]["source_url"],
            "score":      float(similarities[i])
        }
        for i in top_indices
    ]

# ── Compare original vs bonus on test set ────────────────────────────────────
print("\nComparing original (500 words) vs bonus (250 words) chunks...\n")

orig_scores, bonus_scores = [], []

for item in test_set:
    q        = item["question"]
    expected = item["expected"]

    # Original
    orig_chunks  = retrieve(q, top_k=5)
    orig_answer  = generate(q, orig_chunks)
    orig_context = "\n\n".join([c["text"] for c in orig_chunks])
    orig_eval    = evaluate(q, expected, orig_answer, orig_context)

    # Bonus
    bon_chunks  = retrieve_bonus(q, top_k=5)
    bon_answer  = generate(q, bon_chunks)
    bon_context = "\n\n".join([c["text"] for c in bon_chunks])
    bon_eval    = evaluate(q, expected, bon_answer, bon_context)

    orig_scores.append((orig_eval.faithfulness_score, orig_eval.correctness_score))
    bonus_scores.append((bon_eval.faithfulness_score, bon_eval.correctness_score))

    print(f"Q: {q[:60]}...")
    print(f"  Original → Faithfulness: {orig_eval.faithfulness_score}/5  Correctness: {orig_eval.correctness_score}/5")
    print(f"  Bonus    → Faithfulness: {bon_eval.faithfulness_score}/5  Correctness: {bon_eval.correctness_score}/5")
    print()

# ── Summary ───────────────────────────────────────────────────────────────────
print("=" * 60)
print("BEFORE vs AFTER SUMMARY")
print("=" * 60)
print(f"Original (500w) — Avg Faithfulness: {sum(s[0] for s in orig_scores)/len(orig_scores):.2f}  Avg Correctness: {sum(s[1] for s in orig_scores)/len(orig_scores):.2f}")
print(f"Bonus    (250w) — Avg Faithfulness: {sum(s[0] for s in bonus_scores)/len(bonus_scores):.2f}  Avg Correctness: {sum(s[1] for s in bonus_scores)/len(bonus_scores):.2f}")

Re-chunking with smaller chunk size...


Chunking: 100%|██████████| 9560/9560 [00:02<00:00, 4390.37it/s]


Created 28,021 bonus chunks
Embedding bonus chunks...


Embedding: 100%|██████████| 281/281 [10:19<00:00,  2.21s/it]


Saved bonus embeddings: (28021, 1536)

Comparing original (500 words) vs bonus (250 words) chunks...

Q: What programs does the Gabelli School of Business offer?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus    → Faithfulness: 5/5  Correctness: 5/5

Q: How do I apply for financial aid at Fordham?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus    → Faithfulness: 5/5  Correctness: 5/5

Q: Where is Fordham's Rose Hill campus located?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus    → Faithfulness: 5/5  Correctness: 5/5

Q: What is the Lincoln Center campus near?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus    → Faithfulness: 5/5  Correctness: 5/5

Q: What undergraduate research opportunities exist at Fordham?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus    → Faithfulness: 5/5  Correctness: 5/5

Q: What is the summer session undergraduate tuition per credit?...
  Original → Faithfulness: 5/5  Correctness: 5/5
  Bonus   

---

## Git Submission

- [ ] Create a new branch called `homework-5`
- [ ] Commit your work (notebook + Streamlit app + saved artifacts in `temp/`)
- [ ] Push to GitHub
- [ ] Create a Pull Request and merge to main
- [ ] Submit the `.ipynb` file on Blackboard